# CS4120 Final — Part 2, Model 2: Fine-tuned T5 seq2seq

Fine-tune **`t5-small`** (60M params) on the FIREBALL-preprocessed JSON records. We linearize each record to a human-readable prefix via `fireball_preprocess.linearize_for_t5` and train the model to produce the gold narration.

**Why T5 over BART or GPT-2?**
- Data-to-text framing fits T5's text-to-text API naturally.
- `t5-small` fits easily in free-Colab RAM + trains in ~20 min on a T4.
- HuggingFace `Seq2SeqTrainer` gives us BLEU/ROUGE during training for free.

**Runtime on free Colab (T4):** ~20–25 minutes for 3 epochs on ~3.5k examples.

## 1. Install + imports

In [2]:
!pip install -q -U transformers datasets evaluate sacrebleu rouge-score accelerate sentencepiece "numpy>=2.0, <2.1"

In [3]:
import os, sys, json, random
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    set_seed,
)
import evaluate

sys.path.insert(0, '/content')
from fireball_preprocess import linearize_for_t5

SEED = 4120
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

DATA_DIR = Path('/content/processed')
OUT_DIR  = Path('/content/models/t5_small'); OUT_DIR.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cuda


## 2. Load + linearize data

In [4]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = load_jsonl(DATA_DIR/'train.jsonl')
dev_raw   = load_jsonl(DATA_DIR/'dev.jsonl')
test_raw  = load_jsonl(DATA_DIR/'test.jsonl')

def to_pair(r):
    return {'source': linearize_for_t5(r), 'target': r['narration'], 'turn_id': r['turn_id'], 'action_type': r['action_type']}

train_ds = Dataset.from_list([to_pair(r) for r in train_raw])
dev_ds   = Dataset.from_list([to_pair(r) for r in dev_raw])
test_ds  = Dataset.from_list([to_pair(r) for r in test_raw])

print('Example source:\n ', train_ds[0]['source'])
print('Example target:\n ', train_ds[0]['target'])
print()
print('Sizes:', len(train_ds), len(dev_ds), len(test_ds))

Example source:
  narrate | action: attack | actor: Val'trex (Fighter 2/Wizard 9, Githyanki, 107/107 hp) | weapon: flame | recent: Player 3: all good, we can pause, or if u are awake, read above, and we can finish, up to you!) / Player 3: (also val, tauriel, no prep? had time before entering / Player 3: feel free to do prep aswell if u want, had time before entering the warehouse
Example target:
  "Stay back, you were warned, you should have left!"

Sizes: 3533 216 216


## 3. Tokenize

In [5]:
MODEL_NAME = 't5-small'
MAX_SRC = 256
MAX_TGT = 128

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    model_inputs = tokenizer(batch['source'], max_length=MAX_SRC, truncation=True)
    labels = tokenizer(text_target=batch['target'], max_length=MAX_TGT, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
dev_tok   = dev_ds.map(tokenize_fn,   batched=True, remove_columns=dev_ds.column_names)
test_tok  = test_ds.map(tokenize_fn,  batched=True, remove_columns=test_ds.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3533 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

## 4. Model + metrics

In [11]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

bleu   = evaluate.load('sacrebleu')
rouge  = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # clamp to valid vocab range — fast tokenizer's Rust decoder overflows on stray ids                                                                                       │
    preds  = np.clip(preds,  0, tokenizer.vocab_size - 1).astype(np.int64)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1).astype(np.int64)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    return {
        'bleu':   bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])['score'],
        'rougeL': rouge.compute(predictions=decoded_preds, references=decoded_labels)['rougeL'] * 100,
    }

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding='longest')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## 5. Train

In [12]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    predict_with_generate=True,
    generation_max_length=MAX_TGT,
    generation_num_beams=4,
    metric_for_best_model='bleu',
    greater_is_better=True,
    load_best_model_at_end=True,
    report_to='none',
    seed=SEED,
    fp16=(device == 'cuda'),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Rougel
1,4.236238,3.952625,0.271667,8.453160
2,4.042039,3.887004,0.459140,9.420114
3,3.914345,3.879571,0.460952,9.994998


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=663, training_loss=4.119584079241861, metrics={'train_runtime': 193.2864, 'train_samples_per_second': 54.836, 'train_steps_per_second': 3.43, 'total_flos': 642624813563904.0, 'train_loss': 4.119584079241861, 'epoch': 3.0})

## 6. Evaluate on dev + test, save predictions

In [13]:
dev_metrics  = trainer.evaluate(dev_tok,  metric_key_prefix='dev')
test_metrics = trainer.evaluate(test_tok, metric_key_prefix='test')
print('DEV :', dev_metrics)
print('TEST:', test_metrics)

# Generate on test set for 04_evaluate.ipynb
model.eval()
pred_path = OUT_DIR/'predictions_test.jsonl'
records = []
batch = 8
for i in range(0, len(test_raw), batch):
    chunk = test_raw[i:i+batch]
    sources = [linearize_for_t5(r) for r in chunk]
    enc = tokenizer(sources, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SRC).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, max_length=MAX_TGT, num_beams=4, early_stopping=True)
    decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
    for r, p in zip(chunk, decoded):
        records.append({'turn_id': r['turn_id'], 'gold': r['narration'], 'pred': p.strip(), 'action_type': r['action_type']})

with pred_path.open('w') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Wrote:', pred_path, '-', len(records), 'records')

with (OUT_DIR/'results.json').open('w') as f:
    json.dump({
        'model': MODEL_NAME,
        'epochs': args.num_train_epochs,
        'batch_size': args.per_device_train_batch_size,
        'lr': args.learning_rate,
        **{k: float(v) for k, v in dev_metrics.items() if isinstance(v, (int, float))},
        **{k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    }, f, indent=2)
print('Wrote:', OUT_DIR/'results.json')

Training Loss,Validation Loss,Epoch,Bleu,Rougel
3.914345,3.879571,3,0.460952,9.994998


Training Loss,Validation Loss,Epoch,Bleu,Rougel
3.914345,3.852950,3,0.638817,9.307514


DEV : {'dev_loss': 3.8795711994171143, 'dev_bleu': 0.4609519086167526, 'dev_rougeL': 9.994998480779048}
TEST: {'test_loss': 3.852949857711792, 'test_bleu': 0.6388171853007245, 'test_rougeL': 9.307513954490082}
Wrote: /content/models/t5_small/predictions_test.jsonl - 216 records
Wrote: /content/models/t5_small/results.json


## 7. Qualitative samples

In [14]:
for rec in random.Random(0).sample(records, k=6):
    print('ACTION :', rec['action_type'])
    print('GOLD   :', rec['gold'])
    print('T5     :', rec['pred'])
    print('-'*60)

ACTION : attack
GOLD   : It yells at you in infernal before becoming a stain on the bed. The soun of someone dashing up the stairs could be heard almost immediatlely after Vimak would bow his head in respect to the pit fiend before turning to face the ice devils.
T5     : Vimak slams the pit fiend with a sigh of relief.
------------------------------------------------------------
ACTION : attack
GOLD   : The probing wave continues its assault, landing its own blow on the defenders!
T5     : The orc clambers up, spits at Maya's face as it roars!
------------------------------------------------------------
ACTION : attack
GOLD   : Lighter fires off an arrow at one of the ogres... Welp. Goodluck and godspeed Cel. He promptly hides behind a tree.
T5     : Lighter slams the ogre with a longbow and a longbow adv.
------------------------------------------------------------
ACTION : attack
GOLD   : "...Mm" she goes over, looking over Bastiun
T5     : “well, I’m not a fan of this”
------------

## 8. (Optional) Save model to Drive

In [16]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cs4120_dnd/t5_small
!cp -r /content/models/t5_small/* /content/drive/MyDrive/cs4120_dnd/t5_small/

Mounted at /content/drive


## What to take away for the report

- T5 should significantly beat the N-gram on BLEU/ROUGE because it can generalize — proper names from prefix transfer into output.
- Failure modes to discuss: repetition, hallucinated named entities, tendency to regress to short safe narration ("The goblin takes damage.") — especially for rare actions.
- `generation_num_beams=4` is a modest beam; try 8 for more fluent (but slower) outputs.

Sources:
- Raffel et al. (2020) — *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*
- [HuggingFace Seq2SeqTrainer docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainer)